# Speculative Decoding, Derived: Draft Models, N-grams, Medusa and EAGLE

The runnable code of [Speculative Decoding, Derived: Draft Models, N-grams, Medusa and EAGLE](https://residual-stream.lvtan71.workers.dev/posts/speculative-decoding/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/speculative-decoding.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 10 min end to end.
Downloads Qwen2.5-1.5B-Instruct and Qwen2.5-0.5B, about 4 GB.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "speculative-decoding"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['data/blog_val.txt']:
    shutil.copy(os.path.join(ROOT, f), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## The acceptance rule, and why it is exact

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/speculative-decoding/#the-acceptance-rule-and-why-it-is-exact)

In [ ]:
import numpy as np

# 1. The acceptance rule makes the output exactly the target's distribution,
#    whatever the draft. Target p and draft q over 6 tokens: sample from q,
#    accept with probability min(1, p/q), otherwise sample from the residual
#    max(0, p - q), renormalized. Check by sampling a million times.
rng = np.random.default_rng(0)
p = np.array([0.40, 0.25, 0.15, 0.10, 0.07, 0.03])
q = np.array([0.10, 0.10, 0.30, 0.30, 0.10, 0.10])           # a bad draft, on purpose
n = 1_000_000
x = rng.choice(6, n, p=q)
accept = rng.random(n) < np.minimum(1, p[x] / q[x])
resid = np.maximum(p - q, 0); resid /= resid.sum()
x = np.where(accept, x, rng.choice(6, n, p=resid))
print("target p:          ", np.round(p, 3))
print("output frequencies:", np.round(np.bincount(x, minlength=6) / n, 3))
print(f"acceptance rate {accept.mean():.3f}; theory: sum of min(p, q) = {np.minimum(p, q).sum():.3f}")

# 2. Tokens per verification step. If each drafted token is accepted with
#    probability alpha, independently, a step with k drafts yields
#    (1 - alpha^(k+1)) / (1 - alpha) tokens on average: the accepted run,
#    plus the one token the target always contributes (a correction, or a
#    bonus token after k acceptances). Check against simulation.
def simulate(alpha, k, steps=200_000):
    acc = rng.random((steps, k)) < alpha
    run = np.where(acc.all(1), k, acc.argmin(1))              # accepted before the first rejection
    return (run + 1).mean()
print("\nalpha  k   formula  simulated")
for alpha in (0.6, 0.8, 0.9):
    for k in (2, 4, 8):
        f = (1 - alpha ** (k + 1)) / (1 - alpha)
        print(f"{alpha:5.1f} {k:2d} {f:9.2f} {simulate(alpha, k):10.2f}")

# 3. Speedup, counting the draft's cost: c = one draft step's time relative
#    to one target step. A step costs k draft steps plus one target step that
#    verifies all k (decode is memory-bound, so verifying k+1 positions costs
#    about one step).
print("\nspeedup over plain decoding, by draft cost c (alpha = 0.8)")
print("  k " + "".join(f"  c={c:<5g}" for c in (0.0, 0.05, 0.1, 0.3)))
for k in (1, 2, 4, 8, 16):
    f = (1 - 0.8 ** (k + 1)) / (1 - 0.8)
    print(f"{k:3d} " + "".join(f"{f / (k * c + 1):8.2f}" for c in (0.0, 0.05, 0.1, 0.3)))

The post printed:

```text
target p:           [0.4  0.25 0.15 0.1  0.07 0.03]
output frequencies: [0.4  0.25 0.15 0.1  0.07 0.03]
acceptance rate 0.550; theory: sum of min(p, q) = 0.550

alpha  k   formula  simulated
  0.6  2      1.96       1.96
  0.6  4      2.31       2.31
  0.6  8      2.47       2.48
  0.8  2      2.44       2.44
  0.8  4      3.36       3.36
  0.8  8      4.33       4.32
  0.9  2      2.71       2.71
  0.9  4      4.10       4.10
  0.9  8      6.13       6.13

speedup over plain decoding, by draft cost c (alpha = 0.8)
  k   c=0      c=0.05   c=0.1    c=0.3  
  1     1.80    1.71    1.64    1.38
  2     2.44    2.22    2.03    1.52
  4     3.36    2.80    2.40    1.53
  8     4.33    3.09    2.40    1.27
 16     4.89    2.72    1.88    0.84
```

## Measured acceptance

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/speculative-decoding/#measured-acceptance)

In [ ]:
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
DEV = "cuda" if torch.cuda.is_available() else "cpu"  # notebook: use the GPU if there is one

# How often guesses are accepted, measured. Target: Qwen2.5-1.5B-Instruct;
# draft: Qwen2.5-0.5B-Instruct would be the natural pair, but only the
# 0.5B base model is available here, so the draft is the base model (same
# tokenizer). The target writes its answers (greedy, 120 tokens); then, at
# every position of each answer:
#   greedy acceptance: does the draft's top token equal the target's?
#   sampled acceptance at temperature 1: sum over tokens of min(p, q);
#   prompt lookup: take the last 2 tokens, find them earlier in prompt or
#   answer, and guess the token that followed; is it the target's token?
# Two kinds of task: rewriting a given paragraph (most of the answer can be
# copied from the prompt), and open questions (it can't).
T_ID, T_REV = "Qwen/Qwen2.5-1.5B-Instruct", "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
D_ID, D_REV = "Qwen/Qwen2.5-0.5B", "060db6499f32faf8b98477b0a26969ef7d8b9987"
tok = AutoTokenizer.from_pretrained(T_ID, revision=T_REV)
target = AutoModelForCausalLM.from_pretrained(T_ID, revision=T_REV, dtype=torch.bfloat16).eval().to(DEV)
draft = AutoModelForCausalLM.from_pretrained(D_ID, revision=D_REV, dtype=torch.bfloat16).eval().to(DEV)
text = open("blog_val.txt").read()
paras = [p for p in text.split("\n\n") if 400 < len(p) < 900][:6]
tasks = {"rewrite a paragraph": ["Rewrite this paragraph in plain language, keeping its technical terms:\n\n" + p for p in paras],
         "open question": ["What is the KV cache, and why does it grow?", "Explain what a learning rate does.",
                           "Why do language models use tokenizers?", "What is attention in a Transformer?",
                           "How does dropout reduce overfitting?", "What is the difference between a GPU and a CPU?"]}

@torch.no_grad()
def measure(prompts):
    greedy, sampled, lookup, n = 0, 0.0, 0, 0
    for prompt in prompts:
        ids = tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True,
                                      return_tensors="pt", return_dict=True)["input_ids"].to(DEV)
        out = target.generate(ids, max_new_tokens=120, do_sample=False, temperature=None, top_p=None, top_k=None,
                              repetition_penalty=1.0)
        full, start = out, ids.shape[1]
        p = target(full).logits[0, start - 1:-1].float().softmax(-1)
        q = draft(full).logits[0, start - 1:-1].float().softmax(-1)
        nxt = full[0, start:]
        greedy += (q.argmax(-1) == nxt).sum().item()
        sampled += torch.minimum(p, q).sum(-1).sum().item()
        seq = full[0].tolist()
        for i in range(start, len(seq)):
            a, b = seq[i - 2], seq[i - 1]
            guess = next((seq[j + 2] for j in range(i - 3, -1, -1) if seq[j] == a and seq[j + 1] == b), None)
            lookup += guess == seq[i]
        n += len(nxt)
    return greedy / n, sampled / n, lookup / n, n

for name, prompts in tasks.items():
    g, s, l, n = measure(prompts)
    print(f"{name:20s} ({n} tokens): draft model, greedy {g:.0%}, sampled {s:.0%}; prompt lookup {l:.0%}")

The post printed:

```text
rewrite a paragraph  (666 tokens): draft model, greedy 79%, sampled 75%; prompt lookup 42%
open question        (720 tokens): draft model, greedy 72%, sampled 70%; prompt lookup 6%
```

In [ ]:
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
DEV = "cuda" if torch.cuda.is_available() else "cpu"  # notebook: use the GPU if there is one

# The same models, prompts and answers as accept.py, measured the way
# speculation actually runs:
#  - greedy speculation's tokens per step, exactly: with greedy decoding, a
#    run of accepted drafts leaves the draft with the same context as the
#    target, so per-position agreement on the target's text decides every step
#  - acceptance sum min(p, q) on text the target *sampled* at temperature 1
#  - prompt lookup as deployed: match the last 3, else 2, else 1 tokens
#    earlier in the text and propose up to 10 tokens that followed
torch.manual_seed(0)
T_ID, T_REV = "Qwen/Qwen2.5-1.5B-Instruct", "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
D_ID, D_REV = "Qwen/Qwen2.5-0.5B", "060db6499f32faf8b98477b0a26969ef7d8b9987"
tok = AutoTokenizer.from_pretrained(T_ID, revision=T_REV)
target = AutoModelForCausalLM.from_pretrained(T_ID, revision=T_REV, dtype=torch.bfloat16).eval().to(DEV)
draft = AutoModelForCausalLM.from_pretrained(D_ID, revision=D_REV, dtype=torch.bfloat16).eval().to(DEV)
print("vocab", target.config.vocab_size, draft.config.vocab_size, flush=True)
text = open("blog_val.txt").read()
paras = [p for p in text.split("\n\n") if 400 < len(p) < 900][:6]
tasks = {"rewrite": ["Rewrite this paragraph in plain language, keeping its technical terms:\n\n" + p for p in paras],
         "open": ["What is the KV cache, and why does it grow?", "Explain what a learning rate does.",
                  "Why do language models use tokenizers?", "What is attention in a Transformer?",
                  "How does dropout reduce overfitting?", "What is the difference between a GPU and a CPU?"]}


def spec_steps_greedy(match, k):
    # match[i] = draft argmax == target token at answer position i
    i, steps, n = 0, 0, len(match)
    while i < n:
        r = 0
        while r < k and i + r < n and match[i + r]:
            r += 1
        i += r + 1
        steps += 1
    return n / steps


def lookup_propose(seq, i, max_ng=3, num=10):
    for ng in range(max_ng, 0, -1):
        if i - ng < 0:
            continue
        pat = seq[i - ng:i]
        for j in range(i - ng - 1, -1, -1):  # most recent earlier occurrence
            if seq[j:j + ng] == pat:
                return seq[j + ng:min(j + ng + num, i)]  # continuation from known prefix
    return []


def lookup_steps(seq, start, max_ng, num):
    i, steps, proposed_steps, hits_first = start, 0, 0, 0
    while i < len(seq):
        prop = lookup_propose(seq, i, max_ng, num)
        r = 0
        while r < len(prop) and i + r < len(seq) and prop[r] == seq[i + r]:
            r += 1
        if prop:
            proposed_steps += 1
            hits_first += r > 0
        i += r + 1
        steps += 1
    return (len(seq) - start) / steps, proposed_steps / steps, hits_first / max(proposed_steps, 1)


@torch.no_grad()
def measure(prompts):
    rows = []
    for prompt in prompts:
        ids = tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True,
                                      return_tensors="pt", return_dict=True)["input_ids"].to(DEV)
        start = ids.shape[1]
        out = target.generate(ids, max_new_tokens=120, do_sample=False, temperature=None, top_p=None, top_k=None,
                              repetition_penalty=1.0)
        p = target(out).logits[0, start - 1:-1].float().softmax(-1)
        q = draft(out).logits[0, start - 1:-1].float().softmax(-1)
        nxt = out[0, start:]
        match = (q.argmax(-1) == nxt).tolist()
        smin_g = torch.minimum(p, q).sum(-1).mean().item()
        seq = out[0].tolist()
        # original single-token 2-gram metric
        lk = 0
        for i in range(start, len(seq)):
            a, b = seq[i - 2], seq[i - 1]
            guess = next((seq[j + 2] for j in range(i - 3, -1, -1) if seq[j] == a and seq[j + 1] == b), None)
            lk += guess == seq[i]
        # sampled T=1 text
        outs = target.generate(ids, max_new_tokens=120, do_sample=True, temperature=1.0, top_p=1.0, top_k=0,
                               repetition_penalty=1.0)
        ps = target(outs).logits[0, start - 1:-1].float().softmax(-1)
        qs = draft(outs).logits[0, start - 1:-1].float().softmax(-1)
        smin_s = torch.minimum(ps, qs).sum(-1).mean().item()
        rows.append(dict(n=len(match), g=np.mean(match), smin_g=smin_g, smin_s=smin_s, ns=outs.shape[1] - start,
                         lk1=lk / len(match), spec4=spec_steps_greedy(match, 4), spec8=spec_steps_greedy(match, 8),
                         pl=lookup_steps(seq, start, 3, 10), pl2=lookup_steps(seq, start, 2, 10)))
        print({k: (np.round(v, 3) if not isinstance(v, tuple) else tuple(np.round(v, 3))) for k, v in rows[-1].items()}, flush=True)
    return rows


for name, prompts in tasks.items():
    rows = measure(prompts)
    N = sum(r["n"] for r in rows)
    w = lambda key: sum(r[key] * r["n"] for r in rows) / N
    g = w("g")
    print(f"== {name}: n={N}; greedy agree {g:.3f} (per-prompt min {min(r['g'] for r in rows):.2f} max {max(r['g'] for r in rows):.2f})")
    print(f"   smin on greedy text {w('smin_g'):.3f}; smin on T=1 sampled text {np.mean([r['smin_s'] for r in rows]):.3f}")
    print(f"   lookup 1-token 2-gram {w('lk1'):.3f}")
    print(f"   greedy spec tokens/step k=4: actual {np.mean([r['spec4'] for r in rows]):.2f}, iid formula {(1 - g**5)/(1 - g):.2f}; "
          f"k=8 actual {np.mean([r['spec8'] for r in rows]):.2f}, formula {(1 - g**9)/(1 - g):.2f}")
    for key in ("pl", "pl2"):
        print(f"   prompt lookup {key} (tokens/step, frac steps with a proposal, frac proposals with >=1 accepted): "
              f"{tuple(np.round(np.mean([r[key] for r in rows], 0), 3))}", flush=True)

The post printed:

```text
== rewrite: n=666; greedy agree 0.787 (per-prompt min 0.58 max 0.99)
   smin on greedy text 0.754; smin on T=1 sampled text 0.715
   lookup 1-token 2-gram 0.422
   greedy spec tokens/step k=4: actual 3.27, iid formula 3.28; k=8 actual 4.32, formula 4.15
   prompt lookup pl (tokens/step, frac steps with a proposal, frac proposals with >=1 accepted): (np.float64(2.937), np.float64(0.715), np.float64(0.456))
== open: n=720; greedy agree 0.718 (per-prompt min 0.68 max 0.76)
   smin on greedy text 0.702; smin on T=1 sampled text 0.722
   lookup 1-token 2-gram 0.060
   greedy spec tokens/step k=4: actual 2.82, iid formula 2.87; k=8 actual 3.34, formula 3.37
   prompt lookup pl (tokens/step, frac steps with a proposal, frac proposals with >=1 accepted): (np.float64(1.125), np.float64(0.403), np.float64(0.179))
```